In [1]:
print("ok")

ok


In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/opt/anaconda3/envs/medibot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_pdf_file(data):

    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()

    return documents

In [3]:
def text_split(extracted_data):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [4]:
data = load_pdf_file("../Data/")

text_chunks = text_split(data)

print("Number of chunks:", len(text_chunks))

Number of chunks: 40042


In [42]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [43]:
def download_hugging_face_embeddings():

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return embeddings

In [44]:
embeddings = download_hugging_face_embeddings()

print(embeddings)

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False


In [6]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [7]:
query_result

[-0.03447726368904114,
 0.031023191288113594,
 0.006734950002282858,
 0.026109006255865097,
 -0.03936203196644783,
 -0.16030247509479523,
 0.06692396104335785,
 -0.006441459991037846,
 -0.04745053872466087,
 0.014758896082639694,
 0.070875383913517,
 0.05552759766578674,
 0.019193314015865326,
 -0.026251304894685745,
 -0.010109543800354004,
 -0.02694052830338478,
 0.022307414561510086,
 -0.022226668894290924,
 -0.14969265460968018,
 -0.01749309152364731,
 0.007676207460463047,
 0.0543522983789444,
 0.0032544347923249006,
 0.03172598406672478,
 -0.08462139219045639,
 -0.029406001791357994,
 0.05159563943743706,
 0.04812401533126831,
 -0.0033148108050227165,
 -0.058279164135456085,
 0.04196929186582565,
 0.02221071347594261,
 0.128188818693161,
 -0.02233891934156418,
 -0.011656287126243114,
 0.06292840093374252,
 -0.032876305282115936,
 -0.09122607856988907,
 -0.03117530420422554,
 0.052699532359838486,
 0.047034844756126404,
 -0.0842030718922615,
 -0.030056161805987358,
 -0.020744832232

In [45]:
from dotenv import load_dotenv
load_dotenv()

True

In [46]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "medicalbot"

# Check if index exists before creating
if index_name not in pc.list_indexes().names():
    # Create index
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully")
else:
    print(f"Index '{index_name}' already exists")

Index 'medicalbot' already exists


In [51]:
%pip install -U langchain-pinecone pinecone-client

Note: you may need to restart the kernel to use updated packages.


In [57]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [48]:
from langchain_pinecone import Pinecone

In [49]:
docsearch = Pinecone.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [52]:
from langchain_pinecone import Pinecone
docsearch = Pinecone.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [53]:
docsearch

In [54]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [55]:
retrieved_docs = retriever.invoke("What is Acne?")

In [56]:
retrieved_docs

[Document(metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2016-02-07T11:23:03+07:00', 'page': 55, 'page_label': '26', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': '../Data/Medical_book.pdf', 'total_pages': 4505}, page_content='Researchers, Inc. Reproduced by permission.)\n26 GALE ENCYCLOPEDIA OF MEDICINE\nAcne'),
 Document(metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2016-02-07T11:23:03+07:00', 'page': 269, 'page_label': '240', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': '../Data/Medical_book.pdf', 'total_pages': 4505}, page_content='forms of acne.\nPurpose\nDifferent types of antiacne drugs are used for\ndifferent purposes. For example, lotions, soaps, gels,\nand creams containing benzoyl peroxide or tretinoin\nmay be used to clear up mild to moderately severe\nacne. Isotretinoin (Accutane) is prescribed only for\nvery severe, disfiguring acne.\nAcne is a skin condition 

In [58]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.4, max_tokens=500)

In [60]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

# Create the RAG chain
rag_chain = (
    {"context": retriever, "input": RunnablePassthrough()}
    | prompt
    | llm
)

In [64]:
# Invoke the RAG chain to get the answer
try:
    response = rag_chain.invoke("What is Acne?")
    print(response)
except Exception as e:
    print(f"Error: {type(e).__name__}")
    print(f"Details: {str(e)}")
    print("\nNote: Make sure your OpenAI API key is valid and you have sufficient quota.")



Acne is a skin condition that occurs when pores or hair follicles become blocked, allowing a waxy material called sebum to collect inside the pores. Different types of antiacne drugs, such as lotions, soaps, gels, and creams containing benzoyl peroxide or tretinoin, may be used to clear up mild to moderately severe acne. Isotretinoin (Accutane) is prescribed only for very severe, disfiguring acne.
